In [ ]:
import cv2
import numpy as np
import tensorflow as tf
import pandas as pd
import datetime



In [ ]:
# Load pre-trained models
age_model = tf.keras.models.load_model("age_detection_model.h5")
gender_model = tf.keras.models.load_model("gender_detection_model.h5")


In [ ]:
# Define face detector
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

def predict_age_gender(face):
    face = cv2.resize(face, (128, 128)) / 255.0
    face = np.expand_dims(face, axis=0)
    
    age_prediction = age_model.predict(face)
    gender_prediction = gender_model.predict(face)
    
    age = int(age_prediction[0][0])
    gender = "Male" if gender_prediction[0][0] > 0.5 else "Female"
    return age, gender


In [ ]:
# Open webcam
cap = cv2.VideoCapture(0)

# Data storage
data = []

while True:
    ret, frame = cap.read()
    if not ret:
        break
    
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, 1.3, 5)
    
    for (x, y, w, h) in faces:
        face = frame[y:y+h, x:x+w]
        age, gender = predict_age_gender(face)
        
        # Mark senior citizens
        label = f"{gender}, Age: {age}"
        if age > 60:
            label += " (Senior Citizen)"
        
        # Store data
        data.append([age, gender, datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")])
        
        # Display results
        color = (0, 0, 255) if age > 60 else (255, 255, 255)
        cv2.rectangle(frame, (x, y), (x+w, y+h), color, 2)
        cv2.putText(frame, label, (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2)
    
    cv2.imshow("Senior Citizen Identification", frame)
    
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

# Save data to CSV
pd.DataFrame(data, columns=["Age", "Gender", "Timestamp"]).to_csv("senior_citizen_data.csv", index=False)
print("Data saved to senior_citizen_data.csv")